# Notebook 3: Modified CC-ResSiamNet — Training & Inference


In [ ]:
import os, glob, gc, json, math, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

warnings.filterwarnings('ignore')

SEED = 42
SENTINEL_NPZ_DIR  = "./processed_sentinel_npz"
VELOCITY_NPZ_DIR  = "./processed_velocity_npz"
PATCHES_DIR       = "./patches_npy"          # large .npy files for training
MODEL_DIR         = "./model_outputs"
FIGURES_DIR       = "./figures_training"

PATCH_SIZE    = 256
CROP_BORDER   = 16
OUT_SIZE      = PATCH_SIZE - 2 * CROP_BORDER  # 224
PATCH_OVERLAP = 64

V_SCALE       = 5000.0
V_CLIP        = 2.0

BATCH_SIZE    = 16
CHECKPOINT_PATH = os.path.join(MODEL_DIR, 'checkpoint.pt')
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
for d in [PATCHES_DIR, MODEL_DIR, FIGURES_DIR]: os.makedirs(d, exist_ok=True)

print(f"PyTorch: {torch.__version__} | Device: {DEVICE}")
if DEVICE.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}")


## 1) Batch Patchification: NPZ Pairs + Velocity -> Patch .npy Arrays


In [ ]:
from datetime import datetime
import re

def patchify_positions(H, W, patch=PATCH_SIZE, overlap=PATCH_OVERLAP):
    step = patch - overlap
    rows = list(range(0, max(H - patch + 1, 1), step))
    cols = list(range(0, max(W - patch + 1, 1), step))
    if rows and rows[-1] + patch < H: rows.append(H - patch)
    if cols and cols[-1] + patch < W: cols.append(W - patch)
    return [(r, c) for r in rows for c in cols]



def _parse_dates_from_npz(npz_path):
    """
    Extract (date_start, date_end, metadata) from an .npz file.
    Returns (None, None, {}) if the file is corrupt or unreadable.
    """
    try:
        data = np.load(npz_path, allow_pickle=True)
    except Exception as e:
        print(f"  CORRUPT/UNREADABLE: {Path(npz_path).name} ({e})")
        return None, None, {}

    try:
        meta = json.loads(str(data['metadata']))
    except Exception as e:
        print(f"  BAD METADATA: {Path(npz_path).name} ({e})")
        data.close()
        return None, None, {}

    data.close()

    d1_str = meta.get('t1') or meta.get('date_start')
    d2_str = meta.get('t2') or meta.get('date_end')

    if d1_str is None or d2_str is None:
        return None, None, meta

    try:
        d1 = datetime.strptime(d1_str, '%Y-%m-%d')
        d2 = datetime.strptime(d2_str, '%Y-%m-%d')
    except ValueError:
        print(f"  DATE PARSE FAIL: {Path(npz_path).name} ('{d1_str}', '{d2_str}')")
        return None, None, meta

    return d1, d2, meta


def _build_velocity_index(velocity_dir):
    """
    Build a date-indexed list of all valid velocity .npz files.
    Skips corrupt files gracefully.
    Returns: sorted list of dicts with filepath, dates, midpoint.
    """
    vel_files = sorted(glob.glob(os.path.join(velocity_dir, "*_velocity.npz")))
    if not vel_files:
        vel_files = sorted(glob.glob(os.path.join(velocity_dir, "*.npz")))

    print(f"Scanning {len(vel_files)} velocity files in {velocity_dir}...")
    index = []
    skipped = 0

    for vfp in vel_files:
        d1, d2, meta = _parse_dates_from_npz(vfp)
        if d1 is not None and d2 is not None:
            mid = d1 + (d2 - d1) / 2
            index.append({
                'filepath': vfp,
                'date_start': d1,
                'date_end': d2,
                'midpoint': mid,
                'filename': Path(vfp).name,
            })
        else:
            skipped += 1

    index.sort(key=lambda x: x['midpoint'])
    print(f"Velocity index: {len(index)} valid cycles, {skipped} skipped")
    if index:
        print(f"  Date range: {index[0]['date_start'].strftime('%Y-%m-%d')} "
              f"-> {index[-1]['date_end'].strftime('%Y-%m-%d')}")
    return index


def _find_best_velocity_match(sent_d1, sent_d2, vel_index, max_gap_days=18):
    """
    Find the velocity cycle that best matches a Sentinel pair's dates.

    Priority:
    1. Maximum temporal overlap between [sent_d1, sent_d2] and [vel_d1, vel_d2]
    2. If no overlap, closest midpoint (within max_gap_days)
    3. If nothing within max_gap_days, return None
    """
    sent_mid = sent_d1 + (sent_d2 - sent_d1) / 2
    best_entry = None
    best_overlap = -1
    best_gap = float('inf')

    for vel in vel_index:
        overlap_start = max(sent_d1, vel['date_start'])
        overlap_end = min(sent_d2, vel['date_end'])
        overlap_days = (overlap_end - overlap_start).days

        if overlap_days > 0:
            if overlap_days > best_overlap:
                best_overlap = overlap_days
                best_entry = vel
                best_gap = 0
        else:
            gap = abs((sent_mid - vel['midpoint']).days)
            if gap < best_gap and best_overlap <= 0:
                best_gap = gap
                best_entry = vel

    if best_entry is not None and best_overlap <= 0 and best_gap > max_gap_days:
        return None

    return best_entry


def _load_velocity_arrays(vel_filepath):
    """Load vx, vy, valid_mask from a velocity .npz with explicit copy."""
    data = np.load(vel_filepath, allow_pickle=True)
    vx = np.array(data['vx'])
    vy = np.array(data['vy'])
    valid = np.array(data['valid_mask'])
    data.close()
    return vx, vy, valid



def _count_patches_one_file(sfp, vx, vy, valid_vel, min_valid):
    """Count valid patches from one sentinel .npz using its matched velocity."""
    try:
        sdata = np.load(sfp, allow_pickle=True)
        x1 = sdata['x1']
        x2 = sdata['x2']
        smeta = json.loads(str(sdata['metadata']))
        H, W, C = x1.shape
    except Exception as e:
        print(f"  CORRUPT SENTINEL: {Path(sfp).name} ({e})")
        return 0

    if vx.shape == (H, W):
        vmask = valid_vel
    else:
        from scipy.ndimage import zoom
        vmask = zoom(valid_vel.astype(float),
                     (H / vx.shape[0], W / vx.shape[1]), order=0) > 0.5

    s1_any = np.zeros((H, W), dtype=bool)
    ch_names = smeta.get('channels', [])
    for mn in ['s1_valid_asc', 's1_valid_desc']:
        if mn in ch_names:
            idx = ch_names.index(mn)
            s1_any |= (x1[:,:,idx] > 0.5) | (x2[:,:,idx] > 0.5)

    combined = s1_any & vmask
    sdata.close()
    del x1, x2

    count = 0
    b = CROP_BORDER
    for r, c in patchify_positions(H, W):
        m_p = combined[r:r+PATCH_SIZE, c:c+PATCH_SIZE]
        if m_p[b:-b, b:-b].sum() / ((PATCH_SIZE - 2*b)**2) >= min_valid:
            count += 1
    return count



def build_patches_streaming(sentinel_dir, velocity_dir, output_dir,
                            v_scale=V_SCALE, v_clip=V_CLIP, min_valid=0.2,
                            max_gap_days=18):
    """
    True streaming patchification with per-pair velocity date matching.

    For each Sentinel pair (t1, t2), finds the ITS_LIVE velocity cycle
    whose date window overlaps best, and uses THAT specific velocity
    as the ground truth label.

    Two-pass (never holds all patches in RAM):
      Pass 1: match dates + count valid patches per file
      Pass 2: pre-allocate mmap .npy, write patches directly to disk

    Skips if output .npy files already exist.
    """
    needed = [f'{a}_{s}.npy' for a in ['x1','x2','t'] for s in ['train','val','test']]
    if all(os.path.exists(os.path.join(output_dir, f)) for f in needed):
        print(f"Patch .npy files already exist in {output_dir}/ -- SKIPPING.")
        print("  Delete them to force re-run.")
        for s in ['train', 'val', 'test']:
            arr = np.load(os.path.join(output_dir, f'x1_{s}.npy'), mmap_mode='r')
            print(f"  {s}: {arr.shape[0]} patches, shape={arr.shape}")
        return

    sent_files = sorted(glob.glob(os.path.join(sentinel_dir, "PAIR_*.npz")))
    if not sent_files:
        print(f"No Sentinel .npz files in {sentinel_dir}"); return

    vel_index = _build_velocity_index(velocity_dir)
    if not vel_index:
        print(f"No valid velocity .npz files in {velocity_dir}"); return

    N_CH = None
    ch_names = []
    for sfp in sent_files:
        try:
            sample = np.load(sfp, allow_pickle=True)
            smeta0 = json.loads(str(sample['metadata']))
            N_CH = smeta0['n_channels']
            ch_names = smeta0.get('channels', [])
            sample.close()
            break
        except Exception:
            continue
    if N_CH is None:
        print("Could not read any sentinel file!"); return
    print(f"Input channels: {N_CH}")

    print(f"\n--- Pass 1: Date matching + counting ({len(sent_files)} Sentinel files) ---")

    file_info = []  # (filepath, count, date_str, matched_vel_path)
    total = 0
    matched_count = 0
    unmatched_count = 0
    corrupt_count = 0
    vel_cache = {}  # cache: vel_filepath -> (vx, vy, vmask)

    for sfp in tqdm(sent_files, desc="Matching & counting"):
        sent_d1, sent_d2, smeta = _parse_dates_from_npz(sfp)
        if sent_d1 is None:
            corrupt_count += 1
            continue

        vel_match = _find_best_velocity_match(sent_d1, sent_d2, vel_index, max_gap_days)

        if vel_match is None:
            unmatched_count += 1
            if unmatched_count <= 5:
                print(f"  NO MATCH: {Path(sfp).name} "
                      f"({sent_d1.strftime('%Y-%m-%d')}->{sent_d2.strftime('%Y-%m-%d')})")
            elif unmatched_count == 6:
                print(f"  ... (suppressing further no-match messages)")
            continue

        matched_count += 1
        vel_fp = vel_match['filepath']

        if vel_fp not in vel_cache:
            try:
                vx, vy, vmask = _load_velocity_arrays(vel_fp)
                vel_cache[vel_fp] = (vx, vy, vmask)
            except Exception as e:
                print(f"  CORRUPT VELOCITY: {vel_match['filename']} ({e})")
                continue
        vx, vy, vmask = vel_cache[vel_fp]

        cnt = _count_patches_one_file(sfp, vx, vy, vmask, min_valid)
        file_info.append((sfp, cnt, smeta.get('t1', ''), vel_fp))
        total += cnt
        gc.collect()

    del vel_cache
    gc.collect()

    print(f"\nDate matching results:")
    print(f"  Matched:   {matched_count} / {len(sent_files)} sentinel pairs")
    print(f"  Unmatched: {unmatched_count} (no velocity within {max_gap_days} days)")
    print(f"  Corrupt:   {corrupt_count}")
    print(f"  Total valid patches: {total}")

    if total == 0:
        print("\nNo patches! Possible causes:")
        print("  - Date ranges don't overlap between Sentinel and velocity data")
        print("  - Spatial grids don't align (different CRS or extent)")
        print("  - min_valid too high (currently {:.0%})".format(min_valid))
        return

    print(f"\nSample matches:")
    for i in range(min(5, len(file_info))):
        sfp, cnt, dt, vfp = file_info[i]
        print(f"  {Path(sfp).name}  ->  {Path(vfp).name}  ({cnt} patches)")

    file_info.sort(key=lambda x: x[2])

    n_test = int(total * 0.15)
    n_val = int(total * 0.15)
    n_train = total - n_val - n_test
    boundaries = {
        'train': (0, n_train),
        'val': (n_train, n_train + n_val),
        'test': (n_train + n_val, total)
    }
    print(f"\nSplit: train={n_train}, val={n_val}, test={n_test}")

    bytes_total = total * PATCH_SIZE * PATCH_SIZE * (N_CH + N_CH + 2) * 4
    print(f"Estimated disk: {bytes_total/1e9:.1f} GB")

    print(f"\n--- Pre-allocating .npy files ---")
    mmaps = {}
    for sname, (s, e) in boundaries.items():
        n = e - s
        if n == 0: continue
        for aname, chs in [('x1', N_CH), ('x2', N_CH), ('t', 2)]:
            path = os.path.join(output_dir, f'{aname}_{sname}.npy')
            shape = (n, PATCH_SIZE, PATCH_SIZE, chs)
            fp = np.lib.format.open_memmap(path, mode='w+', dtype=np.float32, shape=shape)
            mmaps[(aname, sname)] = fp
            print(f"  {aname}_{sname}: {shape} ({n*PATCH_SIZE*PATCH_SIZE*chs*4/1e9:.2f} GB)")

    print(f"\n--- Pass 2: Writing patches to disk ---")
    global_idx = 0
    split_ptrs = {'train': 0, 'val': 0, 'test': 0}

    _cached_vel_fp = None
    _cached_vx = None
    _cached_vy = None
    _cached_vmask = None

    for sfp, cnt, dt, vel_fp in tqdm(file_info, desc="Writing"):
        if cnt == 0:
            continue

        try:
            sdata = np.load(sfp, allow_pickle=True)
            x1 = np.array(sdata['x1'])
            x2 = np.array(sdata['x2'])
            smeta = json.loads(str(sdata['metadata']))
            sdata.close()
        except Exception as e:
            print(f"  SKIP (pass 2): {Path(sfp).name} ({e})")
            continue

        H, W, C = x1.shape

        if vel_fp != _cached_vel_fp:
            del _cached_vx, _cached_vy, _cached_vmask
            gc.collect()
            _cached_vx, _cached_vy, _cached_vmask = _load_velocity_arrays(vel_fp)
            _cached_vel_fp = vel_fp

        vx_raw, vy_raw, vmask_raw = _cached_vx, _cached_vy, _cached_vmask

        if vx_raw.shape == (H, W):
            vx, vy, vmask = vx_raw, vy_raw, vmask_raw
        else:
            from scipy.ndimage import zoom
            zh, zw = H / vx_raw.shape[0], W / vx_raw.shape[1]
            vx = zoom(vx_raw, (zh, zw), order=1).astype(np.float32)
            vy = zoom(vy_raw, (zh, zw), order=1).astype(np.float32)
            vmask = zoom(vmask_raw.astype(float), (zh, zw), order=0) > 0.5

        target = np.clip(np.stack([vx/v_scale, vy/v_scale], axis=-1), -v_clip, v_clip)

        s1_any = np.zeros((H, W), dtype=bool)
        ch_n = smeta.get('channels', [])
        for mn in ['s1_valid_asc', 's1_valid_desc']:
            if mn in ch_n:
                idx = ch_n.index(mn)
                s1_any |= (x1[:,:,idx] > 0.5) | (x2[:,:,idx] > 0.5)
        combined = s1_any & vmask
        target[~combined] = 0.0

        b = CROP_BORDER
        for r, c in patchify_positions(H, W):
            m_p = combined[r:r+PATCH_SIZE, c:c+PATCH_SIZE]
            if m_p[b:-b, b:-b].sum() / ((PATCH_SIZE - 2*b)**2) >= min_valid:
                for sname, (s_start, s_end) in boundaries.items():
                    if s_start <= global_idx < s_end:
                        wi = split_ptrs[sname]
                        if ('x1', sname) in mmaps:
                            mmaps[('x1', sname)][wi] = x1[r:r+PATCH_SIZE, c:c+PATCH_SIZE]
                            mmaps[('x2', sname)][wi] = x2[r:r+PATCH_SIZE, c:c+PATCH_SIZE]
                            mmaps[('t', sname)][wi]  = target[r:r+PATCH_SIZE, c:c+PATCH_SIZE]
                        split_ptrs[sname] = wi + 1
                        break
                global_idx += 1

        del x1, x2, target, combined
        gc.collect()

    for key, mm in mmaps.items():
        mm.flush()
    del mmaps
    if _cached_vx is not None:
        del _cached_vx, _cached_vy, _cached_vmask
    gc.collect()

    with open(os.path.join(output_dir, 'channel_info.json'), 'w') as f:
        json.dump({
            'channels': ch_names,
            'n_channels': N_CH,
            'v_scale': v_scale,
            'patch_size': PATCH_SIZE,
            'crop_border': CROP_BORDER,
            'max_gap_days': max_gap_days,
            'n_sentinel_files': len(sent_files),
            'n_matched': matched_count,
            'n_unmatched': unmatched_count,
            'splits': {s: (e-st) for s, (st,e) in boundaries.items()}
        }, f, indent=2)

    print(f"\n{'='*60}")
    print(f"DONE. Patches in {output_dir}/")
    for s in ['train', 'val', 'test']:
        p = os.path.join(output_dir, f'x1_{s}.npy')
        if os.path.exists(p):
            a = np.load(p, mmap_mode='r')
            print(f"  {s}: {a.shape[0]} patches ({os.path.getsize(p)/1e9:.2f} GB)")


build_patches_streaming(SENTINEL_NPZ_DIR, VELOCITY_NPZ_DIR, PATCHES_DIR)


## 2) Memory-Mapped Dataset (Handles >100 GB)


In [ ]:
class PatchDataset(Dataset):
    def __init__(self, patches_dir, split='train', normalize=True):
        self.x1 = np.load(os.path.join(patches_dir, f'x1_{split}.npy'), mmap_mode='r')
        self.x2 = np.load(os.path.join(patches_dir, f'x2_{split}.npy'), mmap_mode='r')
        self.t  = np.load(os.path.join(patches_dir, f't_{split}.npy'),  mmap_mode='r')
        self.normalize = normalize
        self.n_channels = self.x1.shape[-1]
        print(f"  {split}: {len(self.x1)} patches, {self.n_channels} channels")
    
    def __len__(self):
        return len(self.x1)
    
    def _prep_input(self, x):
        x = np.array(x, dtype=np.float32)  # copy from mmap
        if self.normalize:
            for c in range(x.shape[-1]):
                ch = x[:, :, c]
                if ch.max() <= 1.5 and ch.min() >= -0.5:
                    continue  # already ~[0,1]
                nz = ch[ch != 0]
                if len(nz) > 10:
                    p1, p99 = np.percentile(nz, [1, 99])
                    if p99 - p1 > 1e-6:
                        x[:, :, c] = np.clip((ch - p1) / (p99 - p1), 0, 1)
        return np.transpose(x, (2, 0, 1))  # (C, H, W)
    
    def _prep_label(self, t):
        t = np.array(t, dtype=np.float32)
        b = CROP_BORDER
        if t.shape[0] == PATCH_SIZE:
            t = t[b:-b, b:-b, :]
        return np.transpose(t, (2, 0, 1))  # (2, 224, 224)
    
    def __getitem__(self, i):
        return (torch.from_numpy(self._prep_input(self.x1[i])),
                torch.from_numpy(self._prep_input(self.x2[i])),
                torch.from_numpy(self._prep_label(self.t[i])))

ch_info_path = os.path.join(PATCHES_DIR, 'channel_info.json')
if os.path.exists(ch_info_path):
    with open(ch_info_path) as f:
        ch_info = json.load(f)
    N_INPUT_CHANNELS = ch_info['n_channels']
    print(f"Channels: {N_INPUT_CHANNELS}")
    print(f"V_SCALE: {ch_info['v_scale']}")
else:
    N_INPUT_CHANNELS = 26  # default from our band list
    print(f"channel_info.json not found, using default: {N_INPUT_CHANNELS}")

train_ds = PatchDataset(PATCHES_DIR, 'train')
val_ds   = PatchDataset(PATCHES_DIR, 'val')
test_ds  = PatchDataset(PATCHES_DIR, 'test')


## 3) CC-ResSiamNet — Multi-Channel Architecture


In [ ]:
class ConvBnAct(nn.Module):
    def __init__(self, ic, oc, k=3, act="swish"):
        super().__init__()
        self.conv = nn.Conv2d(ic, oc, k, padding=k//2, bias=False)
        self.bn = nn.BatchNorm2d(oc)
        self.act = nn.SiLU(True) if act == "swish" else nn.ReLU(True) if act == "relu" else nn.Identity()
    def forward(self, x): return self.act(self.bn(self.conv(x)))

class ResBlock(nn.Module):
    def __init__(self, ic, oc, k=3):
        super().__init__()
        self.s1 = ConvBnAct(ic*2, oc, k, "swish")
        self.s2 = ConvBnAct(oc, oc, k, "swish")
        self.s3 = ConvBnAct(oc, oc, k, act=None)
        self.p1 = ConvBnAct(ic, oc, k, act=None)
        self.p2 = ConvBnAct(ic, oc, k, act=None)
        self.act = nn.SiLU(True)
    def forward(self, x1, x2):
        s = self.s3(self.s2(self.s1(torch.cat([x1, x2], 1))))
        return self.act(self.p1(x1) + s), self.act(self.p2(x2) + s)

class ChAttn(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.ap = nn.AdaptiveAvgPool2d(1); self.mp = nn.AdaptiveMaxPool2d(1)
        h = max(ch//r, 1)
        self.mlp = nn.Sequential(nn.Linear(ch, h), nn.ReLU(True), nn.Linear(h, ch))
    def forward(self, x):
        b, c = x.shape[:2]
        return x * torch.sigmoid(self.mlp(self.ap(x).view(b,c)) + self.mlp(self.mp(x).view(b,c))).view(b,c,1,1)

class SpAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        return x * torch.sigmoid(self.conv(torch.cat([x.mean(1,True), x.max(1,True)[0]], 1)))

class AttnBlock(nn.Module):
    def __init__(self, ic, f=32):
        super().__init__()
        self.pre = nn.Sequential(nn.Conv2d(ic, f, 1, bias=False), nn.BatchNorm2d(f), nn.ReLU(True))
        self.ca = ChAttn(f); self.sa = SpAttn()
        self.post = ConvBnAct(f, f, 1, "relu")
    def forward(self, x): return self.post(self.sa(self.ca(self.pre(x))))

class CC_ResSiamNet(nn.Module):
    def __init__(self, in_ch=1, filters=(32,64,128,256,512,1024),
                 crop=CROP_BORDER, out_ch=2):
        super().__init__()
        self.crop = crop
        self.enc3 = nn.ModuleList(); self.enc1 = nn.ModuleList()
        for i, f in enumerate(filters):
            p = in_ch if i == 0 else filters[i-1]
            self.enc3.append(ResBlock(p, f)); self.enc1.append(ResBlock(f, f))
        self.up = nn.ModuleList([nn.Upsample(scale_factor=2, mode='nearest') for _ in range(len(filters)-1)])
        self.dec3 = nn.ModuleList(); self.dec1 = nn.ModuleList()
        for i in range(len(filters)-1, 0, -1):
            f = filters[i-1]
            self.dec3.append(ResBlock(filters[i]+f, f)); self.dec1.append(ResBlock(f, f))
        self.att = AttnBlock(filters[0]*2, 32)
        self.head = nn.Sequential(ConvBnAct(32, 32, 1, "relu"), nn.Conv2d(32, out_ch, 1))
        self.pool = nn.MaxPool2d(2)

    def forward(self, x1, x2):
        sk = []
        for i in range(len(self.enc3)):
            x1, x2 = self.enc3[i](x1, x2); x1, x2 = self.enc1[i](x1, x2)
            sk.append((x1, x2))
            if i < len(self.enc3)-1: x1, x2 = self.pool(x1), self.pool(x2)
        for s in range(len(self.dec3)):
            s1, s2 = sk[len(sk)-2-s]
            x1, x2 = self.up[s](x1), self.up[s](x2)
            x1 = torch.cat([x1, s2], 1); x2 = torch.cat([x2, s1], 1)
            x1, x2 = self.dec3[s](x1, x2); x1, x2 = self.dec1[s](x1, x2)
        o = self.head(self.att(torch.cat([x1, x2], 1)))
        b = self.crop
        return o[:, :, b:-b, b:-b] if b > 0 else o

class CC_ResSiamNet_Light(CC_ResSiamNet):
    def __init__(self, in_ch=1, crop=CROP_BORDER, out_ch=2):
        super().__init__(in_ch, (32,64,128,256,256,512), crop, out_ch)

USE_LIGHT = False
model = (CC_ResSiamNet_Light if USE_LIGHT else CC_ResSiamNet)(in_ch=N_INPUT_CHANNELS).to(DEVICE)
np_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {'Light' if USE_LIGHT else 'Full'} CC-ResSiamNet(in_ch={N_INPUT_CHANNELS})")
print(f"  Params: {np_:,}")

_x = torch.randn(2, N_INPUT_CHANNELS, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
with torch.no_grad(): _o = model(_x, _x)
print(f"  Shape: {_x.shape} -> {_o.shape}")
del _x, _o


## 4) Loss, Optimizer & Training Loop

In [ ]:
class MaskedHuberLoss(nn.Module):
    def __init__(self, use_mask=True, delta=0.1):
        super().__init__()
        self.use_mask = use_mask; self.delta = delta
    def forward(self, yp, yt):
        ae = torch.abs(yp - yt)
        q = torch.clamp(ae, max=self.delta); l = ae - q
        loss = 0.5 * q**2 + self.delta * l
        if self.use_mask:
            m = (yt.abs().sum(1, True) > 1e-8).float().expand_as(loss)
            n = m.sum((1,2,3)) + 1e-7
            return ((m * loss).sum((1,2,3)) / n).mean()
        return loss.mean()

criterion = MaskedHuberLoss(use_mask=True, delta=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None
print(f"Loss: MaskedHuber(delta=0.1) -> threshold at {0.1*V_SCALE:.0f} m/yr")
print(f"AMP: {'enabled' if scaler else 'disabled (CPU)'}")


In [ ]:
QUICK_TEST = True
if QUICK_TEST:
    EPOCHS, EP_PER_SUB, N_CHUNKS, SUB_SZ = 2, 2, 1, 256
else:
    EPOCHS, EP_PER_SUB, N_CHUNKS, SUB_SZ = 300, 25, 4, None

print(f"QUICK_TEST={QUICK_TEST}: epochs={EPOCHS}, chunks={N_CHUNKS}")


In [ ]:
def save_checkpoint(model, opt, sched, scaler, epoch, best_val, hist, path=CHECKPOINT_PATH):
    """Save full training state for resume."""
    state = {
        'epoch': epoch,
        'best_val': best_val,
        'history': hist,
        'model_state': model.state_dict(),
        'optimizer_state': opt.state_dict(),
        'scheduler_state': sched.state_dict(),
    }
    if scaler is not None:
        state['scaler_state'] = scaler.state_dict()
    torch.save(state, path)


def load_checkpoint(model, opt, sched, scaler, path=CHECKPOINT_PATH):
    """Load training state. Returns (start_epoch, best_val, history)."""
    if not os.path.exists(path):
        return 0, float('inf'), {'tl': [], 'vl': []}

    print(f"Resuming from checkpoint: {path}")
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    opt.load_state_dict(ckpt['optimizer_state'])
    sched.load_state_dict(ckpt['scheduler_state'])
    if scaler is not None and 'scaler_state' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state'])

    ep = ckpt['epoch']
    best = ckpt['best_val']
    hist = ckpt['history']
    print(f"  Resumed at epoch {ep}, best val={best:.6f}")
    return ep, best, hist


def train(model, crit, opt, sched, scaler, train_ds, val_loader):
    """Training with 4-subset rotation + checkpoint resume."""
    start_epoch, best, hist = load_checkpoint(model, opt, sched, scaler)

    if start_epoch >= EPOCHS:
        print(f"Already trained {start_epoch}/{EPOCHS} epochs. Nothing to do.")
        print("  Delete checkpoint to retrain, or increase EPOCHS.")
        return hist

    all_idx = np.arange(len(train_ds))
    np.random.default_rng(SEED).shuffle(all_idx)
    if SUB_SZ: all_idx = all_idx[:SUB_SZ]
    chunks = np.array_split(all_idx, N_CHUNKS)

    done = start_epoch
    rnd = done // EP_PER_SUB  # resume at correct rotation

    print(f"\nTraining: epochs {done+1} -> {EPOCHS}, chunks={N_CHUNKS}, batch={BATCH_SIZE}")

    while done < EPOCHS:
        cid = rnd % N_CHUNKS
        loader = DataLoader(
            torch.utils.data.Subset(train_ds, chunks[cid].tolist()),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

        for _ in range(min(EP_PER_SUB, EPOCHS - done)):
            model.train()
            rl, nbatch = 0.0, 0
            for bx1, bx2, bt in loader:
                bx1, bx2, bt = bx1.to(DEVICE), bx2.to(DEVICE), bt.to(DEVICE)
                opt.zero_grad()
                if scaler:
                    with torch.amp.autocast('cuda'):
                        loss = crit(model(bx1, bx2), bt)
                    scaler.scale(loss).backward()
                    scaler.step(opt)
                    scaler.update()
                else:
                    loss = crit(model(bx1, bx2), bt)
                    loss.backward()
                    opt.step()
                rl += loss.item(); nbatch += 1
            sched.step()

            model.eval()
            vl, vnb = 0.0, 0
            with torch.no_grad():
                for bx1, bx2, bt in val_loader:
                    bx1, bx2, bt = bx1.to(DEVICE), bx2.to(DEVICE), bt.to(DEVICE)
                    if scaler:
                        with torch.amp.autocast('cuda'):
                            vloss = crit(model(bx1, bx2), bt)
                    else:
                        vloss = crit(model(bx1, bx2), bt)
                    vl += vloss.item(); vnb += 1

            tl = rl / max(nbatch, 1)
            vl = vl / max(vnb, 1)
            hist['tl'].append(tl)
            hist['vl'].append(vl)
            done += 1

            if vl < best:
                best = vl
                torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best.pt'))

            if done % 5 == 0:
                save_checkpoint(model, opt, sched, scaler, done, best, hist)

            if done % 5 == 0 or done <= 3 or done == EPOCHS:
                lr = opt.param_groups[0]['lr']
                print(f"Ep {done:3d}/{EPOCHS} [c{cid}] train={tl:.6f} val={vl:.6f} "
                      f"lr={lr:.2e} best={best:.6f}")
        rnd += 1

    torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'final.pt'))
    save_checkpoint(model, opt, sched, scaler, done, best, hist)
    print(f"\nDone. Best val: {best:.6f}")
    return hist


val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
history = train(model, criterion, optimizer, scheduler, scaler, train_ds, val_loader)


## 5) Evaluation & Visualization

In [ ]:
def evaluate(model, loader, v_scale=V_SCALE):
    model.eval()
    errs, spds = [], []
    with torch.no_grad():
        for bx1, bx2, bt in tqdm(loader, desc="Eval"):
            bx1, bx2, bt = bx1.to(DEVICE), bx2.to(DEVICE), bt.to(DEVICE)
            p = model(bx1, bx2).cpu().numpy() * v_scale
            t = bt.cpu().numpy() * v_scale
            m = np.abs(t).sum(1) > 1e-3
            for b in range(p.shape[0]):
                mb = m[b]
                if mb.sum() < 10: continue
                e = np.sqrt((p[b,0,mb]-t[b,0,mb])**2 + (p[b,1,mb]-t[b,1,mb])**2)
                errs.append(e)
                spds.append(np.sqrt(t[b,0,mb]**2 + t[b,1,mb]**2))
    if not errs: print("No valid samples!"); return {}
    errs = np.concatenate(errs); spds = np.concatenate(spds)
    mae = errs.mean(); rmse = np.sqrt((errs**2).mean())
    print(f"Speed MAE={mae:.1f} RMSE={rmse:.1f} Median={np.median(errs):.1f} P90={np.percentile(errs,90):.1f} m/yr")
    print(f"Mean true speed: {spds.mean():.1f} m/yr")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(errs, bins=200, log=True, color='coral', edgecolor='none')
    axes[0].axvline(mae, color='k', ls='--'); axes[0].set_title(f'Error dist (MAE={mae:.0f})')
    axes[0].set_xlabel('Error (m/yr)')
    
    n = min(50000, len(errs)); idx = np.random.choice(len(errs), n, replace=False)
    axes[1].scatter(spds[idx], errs[idx], s=1, alpha=0.05, color='steelblue')
    axes[1].set_xlabel('True speed'); axes[1].set_ylabel('Error'); axes[1].set_title('Error vs Speed')
    
    rel = errs[idx] / np.maximum(spds[idx], 1)
    axes[2].hist(rel[rel < 5], bins=100, color='seagreen', edgecolor='none')
    axes[2].set_title(f'Relative error (median={np.median(rel):.2f})')
    axes[2].set_xlabel('|error| / speed')
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'eval_metrics.png'), dpi=150); plt.show()
    return {'MAE': mae, 'RMSE': rmse}

best_path = os.path.join(MODEL_DIR, 'best.pt')
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    print('Loaded best model')
else:
    print('No best.pt found, using current weights')

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
metrics = evaluate(model, test_loader)


In [ ]:
def plot_samples(model, ds, n=4, v_scale=V_SCALE):
    model.eval()
    fig, axes = plt.subplots(n, 4, figsize=(20, 5*n))
    if n == 1: axes = axes[np.newaxis, :]
    for row, idx in enumerate(np.random.choice(len(ds), n, replace=False)):
        x1, x2, t = ds[idx]
        with torch.no_grad():
            p = model(x1.unsqueeze(0).to(DEVICE), x2.unsqueeze(0).to(DEVICE))
        p = p[0].cpu().numpy() * v_scale; t = t.numpy() * v_scale
        sp = np.sqrt(p[0]**2 + p[1]**2); st = np.sqrt(t[0]**2 + t[1]**2)
        vm = max(st.max(), sp.max(), 1)
        axes[row,0].imshow(st, cmap='magma', vmin=0, vmax=vm); axes[row,0].set_title('True')
        axes[row,1].imshow(sp, cmap='magma', vmin=0, vmax=vm); axes[row,1].set_title('Pred')
        im = axes[row,2].imshow(np.abs(sp-st), cmap='Reds', vmin=0, vmax=vm*0.3)
        axes[row,2].set_title('|Err|'); plt.colorbar(im, ax=axes[row,2])
        axes[row,3].imshow(x1.numpy()[0], cmap='gray'); axes[row,3].set_title('S1 (t1)')
    for ax in axes.flat: ax.axis('off')
    plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR, 'samples.png'), dpi=150); plt.show()

plot_samples(model, test_ds)


In [ ]:
def plot_history(hist):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(hist['tl'], label='Train', alpha=0.8)
    ax.plot(hist['vl'], label='Val', alpha=0.8)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Training History')
    ax.legend(); ax.set_yscale('log')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'training_curve.png'), dpi=150); plt.show()

if 'history' in dir() and history:
    plot_history(history)


## Summary
